# Notebook 2: Feature Engineering and Modeling for a Production Churn System

This notebook teaches the modeling half of the project. It recreates the logic used in the production application and explains not just **what** the code is doing, but **why** each choice makes sense for churn prediction.

The goal is not only to train a model. The goal is to build a system that is reliable, explainable, and usable by non-technical stakeholders.


## What You Will Learn

By working through this notebook, a learner should be able to:

- move from raw customer records to a production-safe feature matrix
- normalize a messy target into a business-friendly churn scale
- understand how the project automatically detects the modeling task type
- engineer lifecycle, activity, value, complaint, and engagement features
- benchmark several models without relying on a single metric blindly
- reason about overfitting controls in a real churn system
- persist the winning model artifact and use it for inference and recommendations


## Notebook Roadmap

We will follow the same sequence used in the application code:

1. Load the raw training data and normalize the target.
2. validate the dataset and detect the modeling strategy.
3. build the engineered feature matrix used in production.
4. inspect learned thresholds and transformed features.
5. train multiple candidate models.
6. compare them with ordinal-aware evaluation metrics.
7. persist the best artifact.
8. use the saved bundle for scoring, explanation, and recommendations.


In [ ]:
from pathlib import Path
import json
import os
import sys
import warnings

import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from IPython.display import Markdown, display

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 120)
pd.set_option("display.max_rows", 200)
pd.set_option("display.float_format", lambda value: f"{value:,.3f}")

def find_project_root(start: Path | None = None) -> Path:
    start = (start or Path.cwd()).resolve()
    for candidate in [start, *start.parents]:
        if (candidate / "config" / "config.json").exists():
            return candidate
    raise FileNotFoundError("Could not locate the churn project root.")

PROJECT_ROOT = find_project_root()
os.chdir(PROJECT_ROOT)
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print(f"Project root resolved to: {PROJECT_ROOT}")


In [ ]:
from sklearn.base import clone

from app.core.config import get_settings
from app.services.predictor import PredictorService
from src.data.load_data import load_train_data
from src.data.preprocess import prepare_training_matrices
from src.data.target_normalization import (
    describe_target_normalization,
    normalize_target_frame,
    resolve_target_normalization_map,
)
from src.data.validate_data import validate_training_frame
from src.models.predict import load_bundle, score_dataframe
from src.models.select_model import compute_selection_score
from src.models.target_manager import TargetManager
from src.models.train import build_candidate_estimators, train_and_select_model
from src.pipelines.training_pipeline import run_training_pipeline

settings = get_settings()
target_column = settings.target_column
normalization_map = resolve_target_normalization_map(settings.target_normalization_map)


## 1. Load the Training Data and Normalize the Target

The production system exposes churn scores from **1** to **5**, where:

- **1** means the customer is least likely to churn
- **5** means the customer is most likely to churn

Normalizing the target at the start keeps training, evaluation, API output, and Streamlit explanations aligned.


In [ ]:
raw_train = load_train_data()
normalized_train = normalize_target_frame(raw_train, target_column, normalization_map)
normalization_summary = describe_target_normalization(
    raw_train[target_column],
    normalized_train[target_column],
    normalization_map,
)

print(f"Raw training shape: {raw_train.shape}")
display(pd.DataFrame([normalization_summary]))
display(
    normalized_train[target_column]
    .value_counts()
    .sort_index()
    .rename_axis("risk_class")
    .reset_index(name="customers")
)


## 2. Validate the Raw Training Frame

Even in a teaching notebook, we should model good professional habits. Validation tells us whether the file is complete enough to trust and whether we should expect modeling to behave like classification or regression.


In [ ]:
validation_report = validate_training_frame(raw_train, target_column)
target_manager_preview = TargetManager().fit(
    normalized_train[target_column],
    original_target=raw_train[target_column],
    normalization_map=normalization_map,
)

display(pd.DataFrame([validation_report.to_dict()]))
display(pd.DataFrame([target_manager_preview.metadata()]))


### Why the Task Is Ordinal Multiclass

A binary churn model would throw away useful structure because class **5** is not equivalent to class **2**. A regression model would also be less natural here because the target is not truly continuous; it is a small ordered set of business tiers.

That is why the project treats the problem as **ordinal multiclass classification** and evaluates models with metrics that respect ordering.


## 3. Build the Same Training Matrix Used in Production

This step is important: the notebook is not creating a special academic preprocessing flow. It is calling the same production code that powers the deployed app.

That makes the notebook a faithful teaching tool and also prevents a common problem where the notebook tells one story while the application runs another.


In [ ]:
feature_builder, target_manager, X, y = prepare_training_matrices(raw_train)

print(f"Engineered feature matrix shape: {X.shape}")
print(f"Target vector shape: {y.shape}")
print(f"Detected task type: {target_manager.task_type}")
print(f"Final churn classes: {target_manager.classes_}")
display(X.head())
display(pd.Series(y).value_counts().sort_index().rename("encoded_count").to_frame())


## 4. Inspect the Engineered Feature Space

The feature builder converts raw fields into a more expressive representation. This includes:

- tenure-derived fields such as `membership_days`, `joining_month`, and `customer_tenure_bucket`
- time-of-day behavior from `last_visit_time`
- activity and complaint flags
- engagement and spend segments
- safe ratio features such as `value_per_login` and `wallet_to_spend_ratio`

A useful teaching habit is to inspect not just the feature matrix, but also the **contract** of the feature builder: which columns are numerical, which are categorical, and which thresholds were learned from the training data.


In [ ]:
feature_summary = feature_builder.get_feature_summary()

feature_group_summary = pd.DataFrame(
    {
        "feature_group": ["numerical", "categorical", "binary", "total"],
        "count": [
            len(feature_summary["numerical_columns"]),
            len(feature_summary["categorical_columns"]),
            len(feature_summary["binary_columns"]),
            len(feature_summary["feature_columns"]),
        ],
    }
)

display(feature_group_summary)
display(pd.DataFrame({"feature": feature_summary["feature_columns"]}).head(40))


In [ ]:
threshold_table = (
    pd.DataFrame.from_dict(feature_summary["thresholds"], orient="index", columns=["value"])
    .rename_axis("threshold_name")
    .reset_index()
)
display(threshold_table)


In [ ]:
raw_example = raw_train.drop(columns=[target_column]).head(1).T.rename(columns={0: "raw_value"})
engineered_example = X.head(1).T.rename(columns={0: "engineered_value"})
transformation_example = raw_example.join(engineered_example, how="outer")
display(transformation_example.head(50))


## 5. Review Candidate Models

The project intentionally trains multiple models instead of assuming one algorithm will always win. This is important when teaching, because learners should see model selection as an evidence-based comparison, not a favorite-algorithm contest.

The candidate pool typically includes:

- logistic regression as a strong linear baseline
- random forest as a robust tree ensemble baseline
- XGBoost as a powerful gradient boosting option
- LightGBM when it is available in the environment


In [ ]:
candidate_estimators = build_candidate_estimators(
    target_manager,
    feature_builder.numerical_columns_ + feature_builder.binary_columns_,
    feature_builder.categorical_columns_,
)

candidate_overview = []
for name, pipeline in candidate_estimators.items():
    model = pipeline.named_steps["model"]
    candidate_overview.append(
        {
            "candidate": name,
            "pipeline_steps": list(pipeline.named_steps.keys()),
            "model_class": model.__class__.__name__,
            "supports_probabilities": hasattr(model, "predict_proba"),
        }
    )

display(pd.DataFrame(candidate_overview))


## 6. How the Notebook Guards Against Overfitting

This is one of the most important teaching sections.

Overfitting happens when a model learns the quirks of the training data too well and loses the ability to generalize. In this project, we reduce that risk through several mechanisms:

- leakage-prone ID and PII columns are removed before modeling
- missing values are handled consistently inside the pipeline
- the validation split is stratified for classification problems
- cross-validation is used on the training portion before holdout comparison
- the final winner is selected with a balanced score, not with accuracy alone
- tree models are regularized through depth, subsampling, and leaf-size controls


## 7. Train and Compare Candidate Models

The helper below performs the full benchmarking cycle:

1. split the data into train and validation folds
2. cross-validate each candidate on the training split
3. fit the model on the full training split
4. evaluate on the holdout validation split
5. compute a selection score that rewards strong ordinal classification behavior


In [ ]:
best_result, candidate_results, holdout_payload = train_and_select_model(
    X,
    y.to_numpy(),
    feature_builder,
    target_manager,
)

comparison_rows = []
for result in candidate_results:
    row = {
        "candidate": result.name,
        "selection_score": round(result.selection_score, 4),
    }
    for metric_name in [
        "accuracy",
        "f1_macro",
        "f1_weighted",
        "quadratic_weighted_kappa",
        "ordinal_mae",
        "log_loss",
        "roc_auc_ovr_weighted",
        "pr_auc_ovr_weighted",
    ]:
        if metric_name in result.validation_metrics:
            row[metric_name] = round(float(result.validation_metrics[metric_name]), 4)
    comparison_rows.append(row)

comparison_df = pd.DataFrame(comparison_rows).sort_values("selection_score", ascending=False)
display(comparison_df)
print(f"Selected model: {best_result.name}")


## 8. Explain the Selection Score

The project does **not** select the winner with only accuracy. For an ordinal churn system, that would be too shallow.

Instead, the selection score combines:

- weighted F1, to reward overall predictive quality across the distribution
- quadratic weighted kappa, to reward ordinal alignment between predicted and true classes
- macro F1, so smaller classes still matter
- ordinal MAE as a penalty, because predicting class 5 when the truth is class 1 is much worse than being one level off


In [ ]:
winning_metrics = best_result.validation_metrics
score_breakdown = pd.DataFrame(
    {
        "component": [
            "f1_weighted",
            "0.25 * quadratic_weighted_kappa",
            "0.10 * f1_macro",
            "-0.05 * ordinal_mae",
            "final_selection_score",
        ],
        "value": [
            float(winning_metrics.get("f1_weighted", 0.0)),
            0.25 * float(winning_metrics.get("quadratic_weighted_kappa", 0.0)),
            0.10 * float(winning_metrics.get("f1_macro", 0.0)),
            -0.05 * float(winning_metrics.get("ordinal_mae", 0.0)),
            compute_selection_score(winning_metrics),
        ],
    }
)
display(score_breakdown)


## 9. Visualize Validation Performance

A confusion matrix is one of the best teaching visuals for ordinal problems because it shows whether errors stay close to the true class or jump across the scale.

For this project, predictions that drift by one level are usually more acceptable than predictions that miss by several levels.


In [ ]:
confusion = np.array(best_result.validation_metrics["confusion_matrix"])
class_labels = [str(label) for label in target_manager.classes_]

fig = px.imshow(
    confusion,
    text_auto=True,
    x=class_labels,
    y=class_labels,
    aspect="auto",
    color_continuous_scale="YlOrBr",
    title=f"Validation Confusion Matrix for {best_result.name}",
    labels={"x": "Predicted class", "y": "True class", "color": "Customers"},
)
fig.show()


## 10. Global Feature Importance

After model selection, we inspect the strongest signals. In teaching contexts, this is where the notebook connects back to the EDA story: do the most important model drivers make business sense?


In [ ]:
best_pipeline = best_result.estimator
preprocessor = best_pipeline.named_steps["preprocessor"]
model = best_pipeline.named_steps["model"]
feature_names = preprocessor.get_feature_names_out()

if hasattr(model, "feature_importances_"):
    importance_values = model.feature_importances_
elif hasattr(model, "coef_"):
    coef = np.asarray(model.coef_)
    importance_values = np.abs(coef).mean(axis=0) if coef.ndim > 1 else np.abs(coef)
else:
    importance_values = np.zeros(len(feature_names))

importance_df = (
    pd.DataFrame({"feature": feature_names, "importance": importance_values})
    .sort_values("importance", ascending=False)
    .head(20)
)
display(importance_df)

fig = px.bar(
    importance_df.sort_values("importance"),
    x="importance",
    y="feature",
    orientation="h",
    title=f"Top Learned Drivers in the {best_result.name} Model",
    color="importance",
    color_continuous_scale="YlOrBr",
)
fig.show()


## 11. Persist the Full Production Artifact

The previous cells trained and compared models interactively. The next cell runs the official project pipeline, which saves the winning bundle and updated metrics artifacts used by the FastAPI and Streamlit layers.

This is the bridge from notebook experimentation to production behavior.


In [ ]:
training_outputs = run_training_pipeline()
display(pd.DataFrame([training_outputs]))


In [ ]:
with open(settings.metrics_path, "r", encoding="utf-8") as file:
    training_summary = json.load(file)

model_comparison_artifact = pd.read_csv(settings.comparison_path)
feature_importance_artifact = pd.read_csv(settings.feature_importance_path)

display(pd.DataFrame([training_summary["task_detection"]]))
display(model_comparison_artifact)
display(feature_importance_artifact.head(20))


## 12. Use the Persisted Bundle for Inference

A high-quality production notebook should always show that the saved bundle can be reloaded and reused independently. This confirms that training and inference are sharing the same transformations.


In [ ]:
bundle = load_bundle()
sample_records = raw_train.sample(3, random_state=settings.random_seed).drop(columns=[target_column])
scored_records = score_dataframe(bundle, sample_records)
display(scored_records)


## 13. Generate Human-Readable Drivers and Recommendations

Prediction alone is not enough for business adoption. The production system translates model output into:

- a normalized risk score
- a risk band
- top risk drivers written in business language
- deterministic retention recommendations tied to observed customer signals


In [ ]:
predictor = PredictorService()
sample_prediction = predictor.predict_record(sample_records.iloc[0].to_dict())
print(json.dumps(sample_prediction, indent=2, default=str))


In [ ]:
probability_breakdown = pd.DataFrame(
    {
        "risk_class": list(sample_prediction["probability_breakdown"].keys()),
        "probability": list(sample_prediction["probability_breakdown"].values()),
    }
)
probability_breakdown["risk_class"] = probability_breakdown["risk_class"].astype(str)

fig = px.bar(
    probability_breakdown,
    x="risk_class",
    y="probability",
    text="probability",
    title="Per-Class Probability Breakdown for One Example Customer",
    color="probability",
    color_continuous_scale="YlOrBr",
)
fig.update_traces(texttemplate="%{text:.3f}", textposition="outside")
fig.update_layout(xaxis_title="Risk class", yaxis_title="Probability")
fig.show()


## 14. Score a Small Batch

Churn systems are rarely used one record at a time. Business teams often upload lists of customers for campaign prioritization, retention triage, or scheduled outreach. The next cell shows how the same bundle can score a mini batch consistently.


In [ ]:
batch_predictions = predictor.predict_batch(sample_records.to_dict(orient="records"))
batch_summary = pd.DataFrame(
    [
        {
            "customer_id": row["customer_id"],
            "predicted_class": row["predicted_class"],
            "risk_band": row["risk_band"],
            "risk_score": row["risk_score"],
            "confidence": row["confidence"],
        }
        for row in batch_predictions
    ]
)
display(batch_summary)


## 15. Practical Ideas for Improving Performance Further

If you wanted to push this project further, strong next steps would include:

- richer cohort features, such as rolling behavioral change instead of only static averages
- more careful threshold tuning for campaign use cases where recall at the high-risk end matters most
- monotonic or ordinal-specific modeling approaches if the business wants even stronger class-order discipline
- calibration analysis focused on the high-risk classes to support decision thresholds
- segment-specific models if different customer types churn for fundamentally different reasons
- post-deployment monitoring for data drift, especially in feedback, complaint status, and engagement behavior


## Final Takeaway

A strong churn solution is not just a fitted model. It is a chain of disciplined decisions:

- define the target in business-friendly terms
- validate and clean the data safely
- engineer behavior and lifecycle features that capture customer health
- compare several models with metrics that reflect business meaning
- persist the winning artifact
- expose predictions in a form that humans can act on

That end-to-end mindset is what turns a notebook project into a portfolio-ready production system.
